# LeetCode 3813 — Vowel-Consonant Score bằng Python

Link đề: https://leetcode.com/problems/vowel-consonant-score/ · Độ khó: **Easy** · Chủ đề: String, Simulation, Counting

Skeleton LeetCode:

```python
class Solution:
    def vowelConsonantScore(self, s: str) -> int:
        ...
```

> Ghi chú: tên bài, đề, ví dụ, constraints và chữ ký hàm (`vowelConsonantScore`) đã được đối chiếu qua WebSearch / leetcode.doocs.org.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: chuỗi `s` gồm chữ cái thường, khoảng trắng và chữ số; đếm số **nguyên âm** `v` (`a, e, i, o, u`) và số **phụ âm** `c` (các chữ cái còn lại), **bỏ qua** khoảng trắng và chữ số; điểm là `floor(v / c)` nếu `c > 0`, ngược lại là `0`.
2. Thành thạo các công cụ đếm ký tự: `str.isalpha()`, toán tử `in` với chuỗi/tập hợp, `sum` trên biểu thức generator, `collections.Counter`.
3. Phân biệt **chia nguyên** `//` với chia thực `/`, và biết xử lý **chia cho 0** bằng điều kiện rõ ràng.
4. Viết 4 cách: một vòng lặp đếm với `if/elif`, biểu thức generator, `Counter`, và mặt nạ bit cho nguyên âm; đối chiếu bằng test ngẫu nhiên có `random.seed`.
5. Tránh các lỗi: chia cho 0, tính khoảng trắng/chữ số là phụ âm, dùng `/` thay `//`, và coi `'y'` là nguyên âm.

## 2. Đề bài & ví dụ

Cho chuỗi `s` gồm chữ cái tiếng Anh thường, khoảng trắng và chữ số.

- **Nguyên âm:** `a, e, i, o, u`.
- **Phụ âm:** mọi chữ cái tiếng Anh thường **không** phải nguyên âm.
- Khoảng trắng và chữ số **không** được tính là nguyên âm hay phụ âm.

Gọi `v` là số nguyên âm và `c` là số phụ âm. **Điểm** của chuỗi là `floor(v / c)` (làm tròn xuống) nếu `c > 0`, ngược lại là `0`. Trả về điểm.

| Ví dụ | s | Output | Giải thích |
|---|---|---|---|
| 1 | `"cooear"` | `2` | `v = 4` (o, o, e, a), `c = 2` (c, r) -> `floor(4/2) = 2` |
| 2 | `"axeyizou"` | `1` | `v = 5`, `c = 3` (x, y, z) -> `floor(5/3) = 1` |
| 3 | `"au 123"` | `0` | Không có phụ âm (`c = 0`) -> điểm `0` |

Constraints: `1 <= s.length <= 100`, `s` chỉ gồm chữ cái thường, khoảng trắng và chữ số.

In [1]:
def count_vc(s):
    v = sum(ch in "aeiou" for ch in s)
    letters = sum(ch.isalpha() for ch in s)
    return v, letters - v

for s, exp in [("cooear", 2), ("axeyizou", 1), ("au 123", 0)]:
    v, c = count_vc(s)
    score = v // c if c > 0 else 0
    print(f"s={s!r:11} v={v} c={c} -> điểm {score} (đề: {exp})")
    assert score == exp

s='cooear'    v=4 c=2 -> điểm 2 (đề: 2)
s='axeyizou'  v=5 c=3 -> điểm 1 (đề: 1)
s='au 123'    v=2 c=0 -> điểm 0 (đề: 0)


## 3. Kiến thức nền

### 3.1. Duyệt chuỗi và `str.isalpha()`

`for ch in s` cho từng ký tự. `ch.isalpha()` là `True` nếu `ch` là chữ cái; `ch.isdigit()` cho chữ số; `ch == ' '` cho khoảng trắng. Ở bài này ta chỉ quan tâm các **chữ cái**.

In [2]:
s = "au 123"
print([ch.isalpha() for ch in s])
print([ch.isdigit() for ch in s], [ch == ' ' for ch in s])

[True, True, False, False, False, False]
[False, False, False, True, True, True] [False, False, True, False, False, False]


### 3.2. Toán tử `in` với chuỗi và tập hợp

`ch in "aeiou"` kiểm tra `ch` có là một ký tự của chuỗi `"aeiou"` không. Dùng `set` (`ch in {'a','e','i','o','u'}`) cũng được và tra cứu `O(1)`. Lưu ý: với một **ký tự đơn** thì hai cách như nhau, nhưng `"ae" in "aeiou"` cũng là `True` (kiểm tra chuỗi con), nên chỉ dùng với từng ký tự.

In [3]:
VOWELS = set("aeiou")
print('e' in "aeiou", 'e' in VOWELS, 'y' in VOWELS, ' ' in VOWELS)
print("ae" in "aeiou")             # chuỗi con! cẩn thận khi kiểm tra nhiều ký tự

True True False False
True


### 3.3. `sum` trên biểu thức generator

`True` đếm như `1`, `False` như `0`, nên `sum(điều_kiện for ch in s)` đếm số ký tự thỏa điều kiện — ngắn gọn thay vì vòng `for` + biến đếm.

In [4]:
s = "cooear"
print(sum(ch in "aeiou" for ch in s))                    # 4
print(sum(1 for ch in s if ch in "aeiou"))              # cách viết tường minh hơn

4
4


### 3.4. Chia nguyên `//`, chia thực `/` và chia cho 0

`7 // 2 = 3` (làm tròn **xuống**), `7 / 2 = 3.5`. Với số không âm, `//` cho đúng `floor`. Chia cho `0` ném `ZeroDivisionError` nên phải kiểm tra `c > 0` **trước** khi chia.

In [5]:
print(5 // 3, 5 / 3, 4 // 2, 0 // 3)
try:
    print(5 // 0)
except ZeroDivisionError as e:
    print("lỗi:", e)

1 1.6666666666666667 2 0
lỗi: integer division or modulo by zero


### 3.5. `collections.Counter`

`Counter(s)` cho từ điển đếm số lần của từng ký tự. Từ đó tổng số nguyên âm là tổng các tần suất của `a, e, i, o, u`.

In [6]:
from collections import Counter
cnt = Counter("axeyizou")
print(cnt)
print(sum(cnt[ch] for ch in "aeiou"))

Counter({'a': 1, 'x': 1, 'e': 1, 'y': 1, 'i': 1, 'z': 1, 'o': 1, 'u': 1})
5


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Bài này là **mô phỏng + đếm**: đi qua từng ký tự và phân loại vào một trong ba nhóm — nguyên âm, phụ âm, "bỏ qua" (khoảng trắng, chữ số). Sau đó tính điểm.

Tính tay ví dụ 2 `"axeyizou"`:

```
ký tự: a x e y i z o u
loại : V C V C V C V V      -> v = 5, c = 3  (x, y, z)
điểm = 5 // 3 = 1
```

Ví dụ 3 `"au 123"`: `a, u` là nguyên âm; `' '`, `1`, `2`, `3` bị bỏ qua; `c = 0` nên điểm `0` (không được chia!).

**Chú ý:** `'y'` **không** là nguyên âm theo đề, nên là phụ âm.

### Bảng phân loại ký tự

| ký tự | loại | đóng góp |
|---|---|---|
| `a e i o u` | nguyên âm | `v += 1` |
| các chữ cái còn lại (`b, c, d, ..., y, z`) | phụ âm | `c += 1` |
| khoảng trắng `' '` | bỏ qua | không |
| chữ số `0..9` | bỏ qua | không |

In [7]:
def classify(ch):
    if ch in "aeiou":
        return "V"
    if ch.isalpha():
        return "C"
    return "-"

for s in ["cooear", "axeyizou", "au 123"]:
    print(f"{s:10} ->", "".join(classify(ch) for ch in s))

cooear     -> CVVVVC
axeyizou   -> VCVCVCVV
au 123     -> VV----


## 5. Cách 1 — Brute Force / baseline

Một vòng `for` qua từng ký tự, rẽ nhánh `if ch in "aeiou"` / `elif ch.isalpha()`. Sau vòng lặp, `return v // c if c > 0 else 0`.

Nút thắt: không có — đây đã là `O(n)`. Các cách sau chỉ khác về **cách viết** (ngắn hơn, ít biến hơn) hoặc kỹ thuật (mặt nạ bit).

In [8]:
class SolutionLoop:
    def vowelConsonantScore(self, s):
        v = c = 0
        for ch in s:
            if ch in "aeiou":
                v += 1
            elif ch.isalpha():
                c += 1
        return v // c if c > 0 else 0

print(SolutionLoop().vowelConsonantScore("cooear"), SolutionLoop().vowelConsonantScore("axeyizou"), SolutionLoop().vowelConsonantScore("au 123"))

2 1 0


**Độ phức tạp:** Time `O(n)`, Space `O(1)`.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước đột phá:** chỉ cần **hai bộ đếm** và một **phép kiểm tra chia cho 0**; thứ tự kiểm tra rất quan trọng: xét nguyên âm **trước**, rồi mới xét "chữ cái còn lại" là phụ âm.

**Quy tắc "nếu… thì…":**

- Nếu `ch` thuộc `"aeiou"` -> nguyên âm (`v += 1`).
- Ngược lại, nếu `ch.isalpha()` -> phụ âm (`c += 1`).
- Ngược lại (khoảng trắng, chữ số) -> bỏ qua.
- Cuối cùng: nếu `c == 0` -> `0`, ngược lại `v // c`.

**Invariant:** sau khi xử lý `k` ký tự đầu, `v` và `c` lần lượt là số nguyên âm và phụ âm trong `k` ký tự đó.

**Biến thể gọn:** `c = (số chữ cái) - v`, vì mọi chữ cái là nguyên âm hoặc phụ âm.

In [9]:
# Kiểm chứng: c = số chữ cái - v, và v + c + (số ký tự bỏ qua) = len(s)
import random
random.seed(4)
ALPHABET = "abcdefghijklmnopqrstuvwxyz 0123456789"
for _ in range(1000):
    s = "".join(random.choice(ALPHABET) for _ in range(random.randint(1, 100)))
    v = sum(ch in "aeiou" for ch in s)
    letters = sum(ch.isalpha() for ch in s)
    skipped = sum(ch == ' ' or ch.isdigit() for ch in s)
    assert v + (letters - v) + skipped == len(s)
print("v + c + (khoảng trắng + chữ số) = len(s)")

v + c + (khoảng trắng + chữ số) = len(s)


## 7. Các cách tiếp theo

### Cách 2 — Biểu thức generator và `sum`

`v = sum(ch in "aeiou" for ch in s)`, `letters = sum(ch.isalpha() for ch in s)`, `c = letters - v`. Ba dòng, không có biến đếm tự cập nhật; nhưng duyệt chuỗi hai lần (vẫn `O(n)`).

In [10]:
class SolutionSum:
    def vowelConsonantScore(self, s):
        v = sum(ch in "aeiou" for ch in s)
        letters = sum(ch.isalpha() for ch in s)
        c = letters - v
        return v // c if c > 0 else 0

print(SolutionSum().vowelConsonantScore("cooear"), SolutionSum().vowelConsonantScore("axeyizou"), SolutionSum().vowelConsonantScore("au 123"))

2 1 0


**Độ phức tạp:** Time `O(n)` (hai lượt), Space `O(1)`.

### Cách 3 — `Counter`

Đếm tần suất bằng `Counter(s)`; `v` là tổng tần suất của năm nguyên âm; `c` là tổng tần suất của các chữ cái (`ch.isalpha()`) trừ `v`. Hợp với những bài cần nhiều thống kê trên cùng chuỗi.

In [11]:
from collections import Counter

class SolutionCounter:
    def vowelConsonantScore(self, s):
        cnt = Counter(s)
        v = sum(cnt[ch] for ch in "aeiou")
        c = sum(freq for ch, freq in cnt.items() if ch.isalpha()) - v
        return v // c if c > 0 else 0

print(SolutionCounter().vowelConsonantScore("cooear"), SolutionCounter().vowelConsonantScore("axeyizou"), SolutionCounter().vowelConsonantScore("au 123"))

2 1 0


**Độ phức tạp:** Time `O(n)`, Space `O(1)` (tối đa 38 khóa khác nhau).

### Cách 4 — Mặt nạ bit cho nguyên âm

Mã hóa tập nguyên âm bằng số nguyên `VOW`: bit thứ `k` bằng `1` nếu chữ cái thứ `k` (`'a' = 0`) là nguyên âm. Với chữ cái `ch`, `(VOW >> (ord(ch) - 97)) & 1` cho `1` nếu nguyên âm. Chỉ áp dụng khi `ch` là chữ cái thường (`'a' <= ch <= 'z'`).

In [12]:
class SolutionMask:
    VOW = sum(1 << (ord(ch) - 97) for ch in "aeiou")

    def vowelConsonantScore(self, s):
        v = c = 0
        for ch in s:
            if 'a' <= ch <= 'z':
                if (self.VOW >> (ord(ch) - 97)) & 1:
                    v += 1
                else:
                    c += 1
        return v // c if c > 0 else 0

print(bin(SolutionMask.VOW))
print(SolutionMask().vowelConsonantScore("cooear"), SolutionMask().vowelConsonantScore("axeyizou"), SolutionMask().vowelConsonantScore("au 123"))

0b100000100000100010001
2 1 0


**Độ phức tạp:** Time `O(n)`, Space `O(1)`. Mặt nạ bit nhanh hơn một chút và cho thấy kỹ thuật "tập con nhỏ bằng số nguyên".

## 8. TRACE từng bước

Trace Cách 1 với `s = "axeyizou"` (mỗi dòng một ký tự):

| bước | ký tự | loại | v | c |
|---|---|---|---|---|
| 1 | a | V | 1 | 0 |
| 2 | x | C | 1 | 1 |
| 3 | e | V | 2 | 1 |
| 4 | y | C | 2 | 2 |
| 5 | i | V | 3 | 2 |
| 6 | z | C | 3 | 3 |
| 7 | o | V | 4 | 3 |
| 8 | u | V | 5 | 3 |

Điểm `= 5 // 3 = 1`.

In [13]:
def trace_loop(s):
    v = c = 0
    print(f"s = {s!r}")
    for i, ch in enumerate(s, 1):
        if ch in "aeiou":
            v += 1
            kind = "V"
        elif ch.isalpha():
            c += 1
            kind = "C"
        else:
            kind = "-"
        print(f"  bước {i}: {ch!r} loại {kind} -> v={v} c={c}")
    score = v // c if c > 0 else 0
    print(f"  điểm = {v} // {c} = {score}" if c > 0 else f"  c = 0 -> điểm = 0")
    return score

assert trace_loop("axeyizou") == 1
assert trace_loop("au 123") == 0

s = 'axeyizou'
  bước 1: 'a' loại V -> v=1 c=0
  bước 2: 'x' loại C -> v=1 c=1
  bước 3: 'e' loại V -> v=2 c=1
  bước 4: 'y' loại C -> v=2 c=2
  bước 5: 'i' loại V -> v=3 c=2
  bước 6: 'z' loại C -> v=3 c=3
  bước 7: 'o' loại V -> v=4 c=3
  bước 8: 'u' loại V -> v=5 c=3
  điểm = 5 // 3 = 1
s = 'au 123'
  bước 1: 'a' loại V -> v=1 c=0
  bước 2: 'u' loại V -> v=2 c=0
  bước 3: ' ' loại - -> v=2 c=0
  bước 4: '1' loại - -> v=2 c=0
  bước 5: '2' loại - -> v=2 c=0
  bước 6: '3' loại - -> v=2 c=0
  c = 0 -> điểm = 0


In [14]:
def trace_mask(s):
    VOW = sum(1 << (ord(ch) - 97) for ch in "aeiou")
    print("VOW =", bin(VOW), "(bit 0=a, 4=e, 8=i, 14=o, 20=u)")
    v = c = 0
    for ch in s:
        if 'a' <= ch <= 'z':
            bit = (VOW >> (ord(ch) - 97)) & 1
            print(f"  {ch!r}: dịch {ord(ch) - 97:>2} bit -> bit = {bit}")
            v += bit
            c += 1 - bit
    return v // c if c > 0 else 0

assert trace_mask("cooear") == 2

VOW = 0b100000100000100010001 (bit 0=a, 4=e, 8=i, 14=o, 20=u)
  'c': dịch  2 bit -> bit = 0
  'o': dịch 14 bit -> bit = 1
  'o': dịch 14 bit -> bit = 1
  'e': dịch  4 bit -> bit = 1
  'a': dịch  0 bit -> bit = 1
  'r': dịch 17 bit -> bit = 0


## 9. Code nộp LeetCode

In [15]:
from typing import List, Optional

class Solution:
    def vowelConsonantScore(self, s: str) -> int:
        v = c = 0
        for ch in s:
            if ch in "aeiou":
                v += 1
            elif ch.isalpha():
                c += 1
        return v // c if c > 0 else 0

## 10. Vì sao thuật toán đúng

- **Phân loại đúng:** mỗi ký tự thuộc đúng một trong ba nhóm rời nhau: nguyên âm, phụ âm, hoặc bỏ qua. Vì ta xét `ch in "aeiou"` **trước**, nên chữ cái là nguyên âm không rơi vào nhánh phụ âm; `elif ch.isalpha()` chỉ bắt các chữ cái còn lại, tức đúng phụ âm; khoảng trắng và chữ số không thỏa cả hai nên bị bỏ qua.
- **Đếm đúng:** mỗi ký tự được xét đúng một lần và tăng đúng một bộ đếm (hoặc không bộ đếm nào), nên cuối cùng `v` và `c` đúng bằng số nguyên âm và phụ âm.
- **Điểm đúng:** đề định nghĩa `floor(v / c)` khi `c > 0`; với số nguyên không âm, `v // c` chính là `floor(v / c)`. Khi `c = 0` ta trả `0` theo đề, và tránh chia cho `0`.
- **Các cách khác:** `letters - v` bằng `c` vì mọi chữ cái là nguyên âm hoặc phụ âm; `Counter` và mặt nạ bit chỉ là hai cách khác nhau để phân loại và đếm, nên cho cùng `v`, `c`.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| Một vòng `for` + `if/elif` | `O(n)` | `O(1)` |
| Hai biểu thức `sum` | `O(n)` (2 lượt) | `O(1)` |
| `Counter` | `O(n)` | `O(1)` (<= 38 khóa) |
| Mặt nạ bit | `O(n)` | `O(1)` |

`n = len(s) <= 100`. Mọi cách đều tuyến tính; ở quy mô này khác biệt chỉ là phong cách.

## 12. Bộ test

Ví dụ của đề, các biên (`n = 1`, chỉ nguyên âm, chỉ phụ âm, chỉ chữ số/khoảng trắng, chuỗi dài 100) và so sánh **tất cả cách** trên nhiều input hợp lệ.

In [16]:
impls = [SolutionLoop(), SolutionSum(), SolutionCounter(), SolutionMask(), Solution()]

cases = [
    ("cooear", 2), ("axeyizou", 1), ("au 123", 0),
    ("a", 0), ("b", 0), ("1", 0), (" ", 0), ("ab", 1), ("ba", 1), ("abb", 0), ("aab", 2),
    ("aeiou", 0), ("bcdfg", 0), ("aeiouaeiou b", 10), ("a" * 100, 0), ("b" * 100, 0),
    ("a" * 50 + "b" * 50, 1), ("a" * 99 + "b", 99), ("12 34 56", 0), ("y", 0), ("ay", 1), ("aay", 2),
]
for s, exp in cases:
    assert 1 <= len(s) <= 100 and all(ch.isalpha() and ch.islower() or ch == ' ' or ch.isdigit() for ch in s)
    for impl in impls:
        got = impl.vowelConsonantScore(s)
        assert got == exp, (type(impl).__name__, s, got, exp)
print("ví dụ + edge OK:", len(cases), "ca")

ví dụ + edge OK: 22 ca


In [17]:
import random
random.seed(3813)
ALPHABET = "abcdefghijklmnopqrstuvwxyz 0123456789"
VOWEL_HEAVY = "aeiou bcd 12"

def oracle(s):
    v = sum(1 for ch in s if ch in "aeiou")
    c = sum(1 for ch in s if ch in "bcdfghjklmnpqrstvwxyz")
    if c == 0:
        return 0
    return v // c

positives = 0
for _ in range(2000):
    n = random.randint(1, 100)
    alpha = random.choice([ALPHABET, VOWEL_HEAVY, "aeiou", "bcd", "ab"])
    s = "".join(random.choice(alpha) for _ in range(n))
    exp = oracle(s)
    positives += exp > 0
    outs = [impl.vowelConsonantScore(s) for impl in impls]
    assert all(o == exp for o in outs), (s, outs, exp)
print("2000 test ngẫu nhiên (seed=3813) OK; số ca điểm > 0:", positives)
assert positives > 300

2000 test ngẫu nhiên (seed=3813) OK; số ca điểm > 0: 547


## 13. Các lỗi người mới hay gặp

### Lỗi 1: chia cho 0 khi không có phụ âm

Ví dụ 3 của đề (`"au 123"`) có `c = 0`. Chia thẳng sẽ ném `ZeroDivisionError`; đề yêu cầu trả `0`.

In [18]:
def wrong_zero_division(s):
    v = sum(ch in "aeiou" for ch in s)
    c = sum(ch.isalpha() for ch in s) - v
    return v // c                                          # SAI: không kiểm tra c > 0

try:
    wrong_zero_division("au 123")
    raised = False
except ZeroDivisionError:
    raised = True
print("ném ZeroDivisionError:", raised, "| đáp án đúng:", Solution().vowelConsonantScore("au 123"))
assert raised and Solution().vowelConsonantScore("au 123") == 0

ném ZeroDivisionError: True | đáp án đúng: 0


### Lỗi 2: tính khoảng trắng và chữ số là phụ âm

Định nghĩa "phụ âm = không phải nguyên âm" mà không lọc chữ cái sẽ đếm cả `' '` và chữ số.

In [19]:
def wrong_not_vowel(s):
    v = sum(ch in "aeiou" for ch in s)
    c = sum(ch not in "aeiou" for ch in s)                 # SAI: gồm cả khoảng trắng và chữ số
    return v // c if c > 0 else 0

print(wrong_not_vowel("ae 1"), "!=", Solution().vowelConsonantScore("ae 1"))
assert wrong_not_vowel("ae 1") == 1 and Solution().vowelConsonantScore("ae 1") == 0

1 != 0


### Lỗi 3: dùng `/` hoặc `round` thay vì `//`

Đề yêu cầu **làm tròn xuống**: `5 / 3 = 1.67`, `round(5 / 3) = 2`, còn đáp án là `1`.

In [20]:
def wrong_round(s):
    v = sum(ch in "aeiou" for ch in s)
    c = sum(ch.isalpha() for ch in s) - v
    return round(v / c) if c > 0 else 0                    # SAI: round thay vì floor

print(wrong_round("axeyizou"), "!=", Solution().vowelConsonantScore("axeyizou"))
assert wrong_round("axeyizou") == 2 and Solution().vowelConsonantScore("axeyizou") == 1

2 != 1


### Lỗi 4: coi `'y'` là nguyên âm

Trong bài này nguyên âm chỉ gồm `a, e, i, o, u`; `'y'` là phụ âm.

In [21]:
def wrong_y_vowel(s):
    v = sum(ch in "aeiouy" for ch in s)                    # SAI: thêm 'y'
    c = sum(ch.isalpha() for ch in s) - v
    return v // c if c > 0 else 0

print(wrong_y_vowel("axeyizou"), "!=", Solution().vowelConsonantScore("axeyizou"))
assert wrong_y_vowel("axeyizou") == 3 and Solution().vowelConsonantScore("axeyizou") == 1

3 != 1


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Đếm ký tự theo loại rồi tính một biểu thức" -> hai bộ đếm + một điều kiện cuối; luôn tự hỏi **mẫu số có thể bằng 0 không**.
- Kiểm tra `ch in "aeiou"` trước, rồi `ch.isalpha()`; thứ tự này tránh đếm trùng.
- `c = (số chữ cái) - v` tránh phải liệt kê 21 phụ âm.
- `//` cho `floor` với số không âm; tránh số thực để khỏi sai số và sai kiểu trả về (đề cần `int`).
- Đọc kỹ "khoảng trắng và chữ số **không tính**": đây là bẫy của phép phủ định "không phải nguyên âm".
- Tự kiểm tra nhanh: `"aeiou"` -> `0` (không có phụ âm), `"ab"` -> `1`, `"abb"` -> `0`.

## 15. Biến thể của bài

1. **Find Most Frequent Vowel and Consonant** (LeetCode 3541): tổng tần suất lớn nhất của một nguyên âm và một phụ âm.
2. **Maximum Number of Vowels in a Substring of Given Length** (1456): cửa sổ trượt đếm nguyên âm.
3. **Reverse Vowels of a String** (345): đảo thứ tự các nguyên âm.
4. **Determine if String Halves Are Alike** (1704): so số nguyên âm hai nửa.
5. **Count Vowel Strings in Range** (2586): đếm từ bắt đầu và kết thúc bằng nguyên âm.
6. Phiên bản có **chữ hoa**: chuyển `s.lower()` trước khi phân loại.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| Vòng `for` + `if/elif` | đếm một lượt | `O(n)` | `O(1)` | Nộp bài, rõ ràng |
| Hai `sum` | `v`, `letters`, `c = letters - v` | `O(n)` | `O(1)` | Viết ngắn |
| `Counter` | đếm tần suất rồi cộng | `O(n)` | `O(1)` | Cần thống kê nhiều thứ cùng lúc |
| Mặt nạ bit | `(VOW >> k) & 1` | `O(n)` | `O(1)` | Luyện kỹ thuật bit, tập ký tự nhỏ |

Nộp cách vòng `for`.

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `Solution().vowelConsonantScore("hello world 42")` ra gì?

**Bài 2 (tính tay):** với `s = "queue 1 sky"`, liệt kê nguyên âm, phụ âm và điểm.

**Bài 3 (điền TODO):** hoàn thành `count_vowels_consonants(s)` trả về cặp `(v, c)`.

**Bài 4 (tự viết lại):** viết `score_upper(s)` cho chuỗi có **cả chữ hoa** (nguyên âm `AEIOUaeiou`), cùng quy tắc điểm.

**Bài 5 (mini challenge):** viết `best_window_score(s, k)` trả điểm lớn nhất của một **đoạn liên tiếp độ dài `k`** của `s`.

In [22]:
# Bài 1: đoán output rồi chạy để kiểm tra
print(Solution().vowelConsonantScore("hello world 42"))

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def count_vowels_consonants_todo(s):
    v = c = 0
    # TODO: duyệt s, tăng v cho nguyên âm, tăng c cho phụ âm, bỏ qua ký tự còn lại
    return v, c

0


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** Bài 1: các chữ cái là h, e, l, l, o, w, o, r, l, d; bỏ qua khoảng trắng và chữ số. Bài 2: `y` là phụ âm.

**Mức 2 (vừa):** Bài 3: giống code nộp nhưng trả về `(v, c)`. Bài 4: kiểm tra `ch in "aeiouAEIOU"` rồi `ch.isalpha()`. Bài 5: duyệt từng `s[i:i+k]` và gọi hàm điểm, hoặc dùng cửa sổ trượt.

**Mức 3 (gần đáp án):** Bài 1: nguyên âm `e, o, o` (3), phụ âm `h, l, l, w, r, l, d` (7), điểm `3 // 7 = 0`. Bài 2: `queue 1 sky`: nguyên âm `u, e, u, e` (4), phụ âm `q, s, k, y` (4), điểm `1`. Bài 5: nhớ rằng `k <= len(s)`.

## 19. Đáp án tham khảo cho bài tập

In [23]:
# Bài 1, 2
assert Solution().vowelConsonantScore("hello world 42") == 0
assert Solution().vowelConsonantScore("queue 1 sky") == 1

# Bài 3
def count_vowels_consonants(s):
    v = c = 0
    for ch in s:
        if ch in "aeiou":
            v += 1
        elif ch.isalpha():
            c += 1
    return v, c
assert count_vowels_consonants("hello world 42") == (3, 7)
assert count_vowels_consonants("queue 1 sky") == (4, 4)
assert count_vowels_consonants("au 123") == (2, 0)

# Bài 4
def score_upper(s):
    v = sum(ch in "AEIOUaeiou" for ch in s)
    c = sum(ch.isalpha() for ch in s) - v
    return v // c if c > 0 else 0
assert score_upper("CoOeAr") == 2 and score_upper("AU 123") == 0 and score_upper("cooear") == Solution().vowelConsonantScore("cooear")

# Bài 5
def best_window_score(s, k):
    return max(Solution().vowelConsonantScore(s[i:i + k]) for i in range(len(s) - k + 1))
assert best_window_score("bbaab", 2) == 1        # 'ba' và 'ab' cho 1 // 1 = 1; 'aa' có c = 0 nên 0
assert best_window_score("aaaa", 2) == 0
assert best_window_score("abb", 3) == 0
assert best_window_score("aab", 3) == 2
print("đáp án bài tập OK")

đáp án bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "đếm theo loại rồi tính tỉ số":**

```python
a = b = 0
for ch in s:
    if ch in "aeiou":        # kiểm tra nhóm đặc biệt trước
        a += 1
    elif ch.isalpha():       # nhóm còn lại của chữ cái
        b += 1
    # còn lại: bỏ qua
return a // b if b > 0 else 0    # kiểm tra mẫu số trước khi chia
```

**Checklist trước khi submit:**

- [ ] Đã kiểm tra `c > 0` trước khi chia (ví dụ `"au 123"` -> `0`)?
- [ ] Khoảng trắng và chữ số không bị tính là phụ âm?
- [ ] Dùng `//` (làm tròn xuống), không dùng `/` hay `round`?
- [ ] `'y'` được xử lý là phụ âm?
- [ ] Thử `"a"`, `"b"`, `"1"`, `"aeiou"`, chuỗi dài 100?
- [ ] Chữ ký đúng `vowelConsonantScore(self, s: str) -> int` và trả `int`?